In [ ]:
import os
import ast
import json
import pickle
import warnings
import numpy as np
import xarray as xr
import proplot as pplt
from matplotlib.lines import Line2D
warnings.filterwarnings('ignore')
pplt.rc.update({
    'savefig.dpi':900,
    'savefig.bbox':'tight',
    'savefig.pad_inches':0.02,
    'tick.minor':False,
    'font.size':9,
    'label.size':9,
    'tick.labelsize':9,
    'title.size':9,
    'abc.size':9,
    'legend.fontsize':9,
    'suptitle.size':9,
    'leftlabelsize':9,
    'toplabelsize':9,
    'leftlabel.weight':'normal',
    'toplabel.weight':'normal',
    'reso':'xx-hi'})

In [ ]:
with open('../scripts/configs.json','r',encoding='utf-8') as f:
    CONFIGS = json.load(f)
SPLITSDIR  = CONFIGS['filepaths']['splits']
WEIGHTSDIR = CONFIGS['filepaths']['weights']
MODELSDIR  = CONFIGS['filepaths']['models']
PREDSDIR   = CONFIGS['filepaths']['predictions']
MODELS     = CONFIGS['experiments']
SPLIT      = 'test'

FIELDVARS    = MODELS['sr']['runs']['sr_gauss']['fieldvars']
NNSEEDS      = MODELS['nn']['seeds']
OPTIMIZEDEQS = MODELS['sr']['optimizedeqs']
PODRUNS      = MODELS['pod']['runs']
NNRUNS       = MODELS['nn']['runs']

SRFUNCTIONS = {
    'cube':lambda x:x**3,
    'square':lambda x:x**2,
    'neg':lambda x:-x,
    'sqrt':np.sqrt,
    'exp':np.exp,
    'log':np.log,
    'abs':np.abs,
    'max':np.maximum,
    'min':np.minimum}

COLORS = {}
LABELS = {}
for name,config in {**PODRUNS,**NNRUNS,**OPTIMIZEDEQS}.items():
    COLORS[name] = config['color']
    LABELS[name] = config['description']

In [ ]:
def kernel_integrate(fields,weights,dsig,mask=None):
    w = fields*weights[None,:,:]*dsig[None,None,:]
    if mask is not None:
        w = w*mask[:,None,:]
    return w.sum(axis=2)

def physical_prediction(output):
    return np.expm1(tpstd*np.maximum(0.0,np.asarray(output,dtype=float)))

def eval_form(form,variables,constants):
    ns = dict(SRFUNCTIONS)
    ns.update(variables)
    ns.update(constants)
    return np.asarray(eval(form,{'__builtins__':{}},ns),dtype=float)

def used_predictors(form,candidates):
    names = {n.id for n in ast.walk(ast.parse(form,mode='eval')) if isinstance(n,ast.Name)}
    return [c for c in candidates if c in names]

def r2(obs,pred):
    mask = np.isfinite(obs) & np.isfinite(pred)
    o,p  = obs[mask],pred[mask]
    return 1 - np.sum((o-p)**2)/np.sum((o-o.mean())**2)

def bin_1d(x,z,nbins=20,minsamples=50,plo=1,phi=99):
    finite = np.isfinite(x)&np.isfinite(z)
    x,z    = x[finite],z[finite]
    lo,hi  = np.percentile(x,[plo,phi])
    edges  = np.linspace(lo,hi,nbins+1)
    xidxs  = np.clip(np.digitize(x,edges)-1,0,nbins-1)
    means  = np.full(nbins,np.nan)
    counts = np.zeros(nbins,dtype=int)
    for i in range(nbins):
        idx = xidxs==i
        counts[i] = idx.sum()
        if counts[i]>=minsamples:
            means[i] = z[idx].mean()
    return 0.5*(edges[:-1]+edges[1:]),means,counts

In [ ]:
with open(os.path.join(SPLITSDIR,'stats.json'),'r',encoding='utf-8') as f:
    STATS = json.load(f)
tpmean = float(STATS['tp_mean'])
tpstd  = float(STATS['tp_std'])

with xr.open_dataset(os.path.join(SPLITSDIR,f'norm_{SPLIT}.h5'),engine='h5netcdf') as ds:
    ntime = ds.sizes['time']
    nlat  = ds.sizes['lat']
    nlon  = ds.sizes['lon']
    nsig  = ds.sizes.get('sig',1)
    lat   = ds['lat'].values
    lon   = ds['lon'].values
    dsig  = ds['dsig'].values
    farrs      = [ds[v].transpose('time','lat','lon','sig').values.reshape(-1,nsig) for v in FIELDVARS]
    fieldstack = np.stack(farrs,axis=1)
    surfmask   = (ds['surfmask'].transpose('time','lat','lon','sig').values.reshape(-1,nsig)
                  if 'surfmask' in ds else None)
    def getflat(da):
        if 'time' in da.dims:
            return da.transpose('time','lat','lon').values.ravel()
        return np.tile(da.values,(ntime,1,1)).ravel()
    blnorm  = getflat(ds['bl'])
    lfnorm  = getflat(ds['lf'])
    shfnorm = getflat(ds['shf'])
    lhfnorm = getflat(ds['lhf'])

kwlist = []
for seed in NNSEEDS:
    wpath = os.path.join(WEIGHTSDIR,f'nn_gauss_{seed}_weights.nc')
    if os.path.exists(wpath):
        with xr.open_dataset(wpath,engine='h5netcdf') as wds:
            kwlist.append(wds['k'].values)
ki = np.mean([kernel_integrate(fieldstack,kw,dsig,surfmask) for kw in kwlist],axis=0) if kwlist else fieldstack.mean(axis=2)
rhk,thetaek,thetaestark = ki[:,0],ki[:,1],ki[:,2]

with xr.open_dataset(os.path.join(SPLITSDIR,f'{SPLIT}.h5'),engine='h5netcdf') as ds:
    obsflat = ds.tp.transpose('time','lat','lon').values.ravel()
    months  = np.tile(
        ds.time.dt.month.values[:,None,None]*np.ones((1,nlat,nlon)),
        1).ravel().astype(int)
    lfraw   = getflat(ds['lf'])
    shfraw  = getflat(ds['shf'])

# lat/lon index arrays for every sample
latidx = np.tile(np.arange(nlat)[None,:,None],(ntime,1,nlon)).ravel()
lonidx = np.tile(np.arange(nlon)[None,None,:],(ntime,nlat,1)).ravel()
lats   = lat[latidx]
lons   = lon[lonidx]

with open(os.path.join(MODELSDIR,'sr','optimized_equations.pkl'),'rb') as f:
    SR_REGISTRY = pickle.load(f)

In [ ]:
VARS = {'bl':blnorm,'rh':rhk,'thetae':thetaek,'thetaestar':thetaestark,
        'lf':lfnorm,'shf':shfnorm,'lhf':lhfnorm}

MODELPRED = {}
for name,cfg in OPTIMIZEDEQS.items():
    entry     = SR_REGISTRY.get(name,{})
    form      = entry.get('form',cfg['form'])
    constants = entry.get('constants',cfg['init'])
    pnames    = used_predictors(form,VARS.keys())
    variables = {p:VARS[p] for p in pnames}
    MODELPRED[name] = physical_prediction(eval_form(form,variables,constants))

def load_predictions(name,split=SPLIT):
    path = os.path.join(PREDSDIR,f'{name}_{split}_predictions.nc')
    if not os.path.exists(path): return None
    with xr.open_dataset(path,engine='h5netcdf') as ds:
        da = ds['tp'].load()
    if 'seed' in da.dims: da = da.mean('seed')
    pred = da.transpose('time','lat','lon').values.ravel()
    return pred if pred.shape[0]==obsflat.shape[0] else None

for name,cfg in {**PODRUNS,**NNRUNS}.items():
    pred = load_predictions(name)
    if pred is not None:
        MODELPRED[name] = pred

valid = np.isfinite(obsflat)
for arr in VARS.values():
    valid &= np.isfinite(arr)

print('Loaded:', sorted(MODELPRED.keys()))
for name,pred in MODELPRED.items():
    print(f'  {LABELS.get(name,name):20s}  R²={r2(obsflat[valid],pred[valid]):.3f}')

## 1. What does NN-GAUSS know that SR-HI doesn't?
If we plot SR-HI's residuals against the NN-GAUSS prediction, a strong relationship means the NN is capturing variance that SR-HI leaves on the table.

In [ ]:
def plot_residual_vs_nn(srname='sr_hi', nnname='nn_gauss', nbins=25):
    obs  = obsflat[valid]
    srp  = MODELPRED[srname][valid]
    nnp  = MODELPRED[nnname][valid]
    resid = obs - srp

    xc,rbin,_ = bin_1d(nnp, resid, nbins=nbins, plo=1, phi=99)
    xc2,obin,_ = bin_1d(nnp, obs,  nbins=nbins, plo=1, phi=99)
    xc3,sbin,_ = bin_1d(nnp, srp,  nbins=nbins, plo=1, phi=99)

    fig,axs = pplt.subplots(nrows=1,ncols=2,refwidth=2.5,share=False)
    axs[0].axhline(0,color='gray',linewidth=0.8,linestyle='--')
    axs[0].plot(xc,rbin,color='k',linewidth=1.5)
    axs[0].format(xlabel=f'{LABELS[nnname]} Prediction (mm)',
                  ylabel=f'Observed $-$ {LABELS[srname]} (mm)',
                  title='SR-HI Residual vs. NN-GAUSS Prediction',grid=False)

    axs[1].plot(xc2,obin,color='k',linewidth=1.5,label='Observed')
    axs[1].plot(xc3,sbin,color=COLORS[srname],linewidth=1.5,label=LABELS[srname])
    axs[1].plot(xc,xc,color=COLORS[nnname],linewidth=1.5,linestyle='--',label=f'{LABELS[nnname]} (1:1)')
    axs[1].format(xlabel=f'{LABELS[nnname]} Prediction (mm)',
                  ylabel='Precipitation (mm)',
                  title='Obs & SR-HI vs. NN-GAUSS',grid=False)
    axs[1].legend(loc='ul',ncols=1)
    pplt.show()

plot_residual_vs_nn()

## 2. Conditional R² by precipitation intensity
Where in the distribution does the R² gap open up?

In [ ]:
def plot_conditional_r2(names=None, nbins=15, minsamples=200):
    if names is None:
        names = [n for n in ['sr_hi','nn_gauss','sr_med','sr_lo','nn_bl'] if n in MODELPRED]
    obs   = obsflat[valid]
    edges = np.percentile(obs, np.linspace(0,95,nbins+1))
    xc    = 0.5*(edges[:-1]+edges[1:])
    idxs  = np.clip(np.digitize(obs,edges)-1,0,nbins-1)

    fig,ax = pplt.subplots(refwidth=3,refheight=2)
    ax.axhline(0,color='gray',linewidth=0.8,linestyle='--')
    for name in names:
        pred   = MODELPRED[name][valid]
        r2bins = []
        for i in range(nbins):
            idx = idxs==i
            if idx.sum()<minsamples:
                r2bins.append(np.nan)
            else:
                r2bins.append(r2(obs[idx],pred[idx]))
        ax.plot(xc,r2bins,color=COLORS[name],linewidth=1.5,label=LABELS[name])
    ax.format(xlabel='Observed Precipitation (mm)',ylabel='R²',
              title='Conditional R² by Intensity Bin',grid=False)
    ax.legend(loc='ur',ncols=1)
    pplt.show()

plot_conditional_r2()

## 3. R² by month and geographic subregion
Does the gap vary by monsoon phase (June / July / August) or geography (land vs ocean, western vs eastern domain)?

In [ ]:
def plot_r2_by_month(names=None):
    if names is None:
        names = [n for n in ['sr_hi','nn_gauss','sr_med','nn_bl'] if n in MODELPRED]
    obs  = obsflat[valid]
    mons = months[valid]
    month_labels = {6:'June',7:'July',8:'August'}

    fig,ax = pplt.subplots(refwidth=3,refheight=2)
    x = np.array([6,7,8])
    for name in names:
        pred = MODELPRED[name][valid]
        r2s  = [r2(obs[mons==m],pred[mons==m]) for m in [6,7,8]]
        ax.plot(x,r2s,color=COLORS[name],linewidth=1.5,marker='o',markersize=4,label=LABELS[name])
    ax.format(xlabel='Month',ylabel='R²',xticks=[6,7,8],
              xticklabels=['June','July','August'],
              title='R² by Month',grid=False)
    ax.legend(loc='b',ncols=2)
    pplt.show()

def plot_r2_by_region(names=None):
    if names is None:
        names = [n for n in ['sr_hi','nn_gauss','sr_med','nn_bl'] if n in MODELPRED]
    obs  = obsflat[valid]
    lf_v = lfraw[valid]
    la_v = lats[valid]
    lo_v = lons[valid]

    regions = {
        'Ocean':   lf_v < 0.1,
        'Land':    lf_v > 0.9,
        'Lat 5-15': (la_v>=5)  & (la_v<15),
        'Lat 15-25':(la_v>=15) & (la_v<=25),
        'Lon 60-75':(lo_v>=60) & (lo_v<75),
        'Lon 75-90':(lo_v>=75) & (lo_v<=90),
    }

    fig,ax = pplt.subplots(refwidth=4,refheight=2)
    rnames = list(regions.keys())
    x      = np.arange(len(rnames))
    width  = 0.8/len(names)
    for i,name in enumerate(names):
        pred = MODELPRED[name][valid]
        r2s  = [r2(obs[mask],pred[mask]) for mask in regions.values()]
        ax.bar(x+i*width,r2s,width=width,color=COLORS[name],label=LABELS[name],alpha=0.85)
    ax.format(xlabel='',ylabel='R²',xticks=x+0.4,xticklabels=rnames,
              title='R² by Geographic Subregion',grid=False,xrotation=20)
    ax.legend(loc='ur',ncols=1)
    pplt.show()

plot_r2_by_month()
plot_r2_by_region()

## 4. What predicts the gap? Binned SR-HI residuals vs. all input features
Bin the (NN-GAUSS − SR-HI) difference against every available input variable. Large slopes = variables carrying information SR-HI doesn't use.

In [ ]:
def plot_gap_vs_features(srname='sr_hi', nnname='nn_gauss'):
    obs  = obsflat[valid]
    gap  = MODELPRED[nnname][valid] - MODELPRED[srname][valid]  # positive = NN predicts more
    resid = obs - MODELPRED[srname][valid]

    features = {
        'RH (KI)':      rhk[valid],
        r'$\theta_e$ (KI)':   thetaek[valid],
        r'$\theta_e^*$ (KI)': thetaestark[valid],
        'LF (norm)':    lfnorm[valid],
        'SHF (norm)':   shfnorm[valid],
        'LHF (norm)':   lhfnorm[valid],
        'BL (norm)':    blnorm[valid],
    }

    ncols = 4
    nrows = -(-len(features)//ncols)
    fig,axs = pplt.subplots(nrows=nrows,ncols=ncols,refwidth=1.5,sharex=False,sharey=False)
    for ax,( fname,fvals) in zip(axs,features.items()):
        xc,gbin,_ = bin_1d(fvals,gap,   nbins=20,plo=1,phi=99)
        xc,rbin,_ = bin_1d(fvals,resid, nbins=20,plo=1,phi=99)
        ax.axhline(0,color='gray',linewidth=0.8,linestyle='--')
        ax.plot(xc,rbin,color=COLORS[srname],linewidth=1.5,label=f'Obs$-${LABELS[srname]}')
        ax.plot(xc,gbin,color=COLORS[nnname],linewidth=1.5,label=f'{LABELS[nnname]}$-${LABELS[srname]}')
        ax.format(xlabel=fname,grid=False)
    for ax in axs[len(features):]:
        ax.set_visible(False)
    axs[:,0].format(ylabel='mm')
    handles = [
        Line2D([],[],color=COLORS[srname],linewidth=1.5,label=f'Obs $-$ {LABELS[srname]}'),
        Line2D([],[],color=COLORS[nnname],linewidth=1.5,label=f'{LABELS[nnname]} $-$ {LABELS[srname]}'),
    ]
    fig.legend(handles,loc='b',ncols=2)
    pplt.show()

plot_gap_vs_features()

## 5. Are there samples NN-GAUSS gets right that SR-HI systematically misses?
Partition samples by which model is closer to observed. Characterize those samples.

In [ ]:
def plot_nn_wins(srname='sr_hi', nnname='nn_gauss', thresh_mm=1.0):
    obs  = obsflat[valid]
    srp  = MODELPRED[srname][valid]
    nnp  = MODELPRED[nnname][valid]

    sr_err = np.abs(obs-srp)
    nn_err = np.abs(obs-nnp)

    # NN wins: NN is at least thresh_mm closer to obs
    nn_wins = (sr_err - nn_err) > thresh_mm
    sr_wins = (nn_err - sr_err) > thresh_mm
    both_ok = (~nn_wins) & (~sr_wins)

    print(f'NN wins on {nn_wins.mean()*100:.1f}% of valid samples')
    print(f'SR wins on {sr_wins.mean()*100:.1f}% of valid samples')
    print(f'Comparable on {both_ok.mean()*100:.1f}%')

    features = {
        'Obs precip (mm)':   obs,
        'RH (KI)':           rhk[valid],
        r'$\theta_e$ (KI)':  thetaek[valid],
        'LF (norm)':         lfnorm[valid],
        'SHF (norm)':        shfnorm[valid],
    }

    fig,axs = pplt.subplots(nrows=1,ncols=len(features),refwidth=1.5,sharex=False,sharey=False)
    for ax,(fname,fvals) in zip(axs,features.items()):
        lo,hi = np.percentile(fvals,[2,98])
        kw = dict(bins=30,range=(lo,hi),density=True,alpha=0.6)
        ax.hist(fvals[nn_wins], color=COLORS[nnname], label='NN wins', **kw)
        ax.hist(fvals[sr_wins], color=COLORS[srname], label='SR wins', **kw)
        ax.format(xlabel=fname,grid=False,ylabel='Density')
    axs[-1].legend(loc='ur',ncols=1)
    fig.format(title=f'Distribution of inputs where each model wins (|err| gap > {thresh_mm} mm)')
    pplt.show()

plot_nn_wins(thresh_mm=0.5)

## 6. How much of the gap is irreducible noise?
Upper bound on predictability: compare both models' R² against a persistence baseline and against the variance of precipitation itself.

In [ ]:
def plot_variance_decomposition(names=None):
    if names is None:
        names = [n for n in ['sr_bl','sr_lo','sr_med','sr_hi','nn_bl','nn_gauss'] if n in MODELPRED]
    obs   = obsflat[valid]
    total_var = np.var(obs)

    # Climatological baseline: predict the mean at each grid point
    obs3d    = obsflat.reshape(ntime,nlat,nlon)
    clim_map = np.nanmean(obs3d,axis=0,keepdims=True)*np.ones((ntime,1,1))
    climflat = clim_map.ravel()[valid]
    r2_clim  = r2(obs,climflat)

    r2s = {'Climatology': r2_clim}
    for name in names:
        r2s[LABELS[name]] = r2(obs, MODELPRED[name][valid])

    labels = list(r2s.keys())
    vals   = list(r2s.values())
    colors = ['gray'] + [COLORS[n] for n in names]

    fig,ax = pplt.subplots(refwidth=4,refheight=2)
    x = np.arange(len(labels))
    ax.bar(x,vals,color=colors,alpha=0.85)
    ax.format(ylabel='R²',xticks=x,xticklabels=labels,xrotation=25,
              title='Overall R² — Model Ladder',grid=False,ylim=(0,1))
    pplt.show()
    for k,v in r2s.items():
        print(f'  {k:25s}  R²={v:.3f}')

plot_variance_decomposition()